## 1. 環境建置與安裝
安裝專案所需的 Python 核心套件。

本區塊執行重點包含：
1. **相容性版本鎖定**：保留原專題使用的 `Gradio 4.44.1`，並固定相容的 `FastAPI` / `Starlette` / `Pydantic` 組合，避免新版 Colab web stack 與舊版 Gradio 的 `TemplateResponse` API 衝突。
2. **核心套件**：安裝 `LangChain`（邏輯控制）、`FAISS`（向量資料庫）與 `PyPDF`（文件讀取），建構純文字的知識檢索環境。


In [ ]:
import os
import sys
from google.colab import userdata

# 1. 安裝全套環境 (含聯網搜尋功能)
print("正在建構全能家教環境 (Gradio 4.44.1 + compatible web stack + Search)...")

# 安裝
!pip install "gradio==4.44.1" "fastapi==0.112.2" "starlette==0.38.6" "pydantic==2.9.2" "jinja2==3.1.5" langchain-groq langchain-community langchain-huggingface faiss-cpu pypdf transformers accelerate "huggingface_hub<1.0" duckduckgo-search

# 2. 設定 Groq 金鑰
try:
    if "GROQ_API_KEY" not in os.environ:
        os.environ["GROQ_API_KEY"] = userdata.get('Groq')
    print("✅ 環境設定完成！")
except Exception as e:
    print(f"❌ 金鑰讀取失敗: {e}")

LLM_MODEL_NAME = "openai/gpt-oss-120b"

## 2. 載入核心模型與工具 (Module)
這一步是建立 AI 的「知識處理核心」，專注於文件理解：
1. **Embedding 模型**：載入 HuggingFace 模型，將 PDF 講義文字轉化為向量，讓 AI 能「讀懂」講義。
2. **PDF 處理器**：定義 `process_uploaded_pdf` 函數，負責將使用者上傳的檔案切分並存入向量庫 (RAG)。

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 初始化裝置
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用裝置: {device}")

# 2. 載入 Embedding 模型 (用於讀取 PDF)
print("正在載入 Embedding 模型...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. PDF 處理函數
def process_uploaded_pdf(file_path):
    if not file_path: return None
    try:
        print(f"📄 正在處理檔案: {file_path}...")
        loader = PyPDFLoader(file_path)
        documents = loader.load()

        # 切分文本
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        texts = text_splitter.split_documents(documents)

        # 建立向量庫
        vectorstore = FAISS.from_documents(texts, embedding_model)
        # 設定檢索器
        retriever = vectorstore.as_retriever(search_kwargs={"k": 4}) # 增加檢索段落數以提升準確度

        print(f"✅ PDF 處理完成，共 {len(texts)} 段。")
        return retriever
    except Exception as e:
        print(f"❌ PDF 錯誤: {e}")
        return None

print("✅ 核心工具組準備完成！")

## 3. 設定 AI 大腦與測驗邏輯 (Prompts)
這一步設定 LLM 的人設、記憶機制與互動測驗邏輯：
1. **全能家教人設**：設定 System Prompt，要求 AI 優先依據講義回答，並保持客觀、清晰的純文字教學風格。
2. **對話記憶 (Memory)**：在對話中注入歷史紀錄，讓 AI 能記得上下文 (例如追問「那另一個呢？」)。
3. **測驗系統**：預先定義「出題老師」與「批改老師」的 Prompt 模板，讓 AI 能根據剛才的對話自動生成題目並批改使用者的回答。

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from duckduckgo_search import DDGS

# 1. 初始化 LLM
llm = ChatGroq(
    model_name=LLM_MODEL_NAME,
    temperature=0.3,
    max_tokens=2048
)

# 2. 搜尋工具
def search_web_text(query):
    try:
        results = DDGS().text(query, max_results=3)
        if results: return "\n".join([f"- {r['body']}" for r in results])
    except: pass
    return None

# 3. 意圖判斷 Prompt
intent_prompt = """你是一個對話意圖分類器。
請判斷使用者的輸入屬於哪一類：

1. **WANT_QUIZ**：使用者想要測驗、出題、挑戰，或同意進行測驗。
2. **REFUSE_QUIZ**：使用者明確拒絕測驗，或不想回答。
3. **OTHER**：其他一般對話 (問問題、閒聊、打招呼)。

使用者輸入："{user_input}"

請只回傳標籤：WANT_QUIZ, REFUSE_QUIZ, 或 OTHER。
"""
intent_chain = ChatPromptTemplate.from_messages([("system", intent_prompt)]) | llm | StrOutputParser()

# 4. 主教學 System Prompt (已移除行事曆功能)
system_prompt = """你是一個專業、客觀的 AI 全科家教。

【回答策略】
1. **優先順序**：優先依據【參考資料】(PDF/搜尋)，其次才用內建知識。
2. **避免幻覺**：不知為不知。
3. **純文字教學**：禁止輸出圖片生成指令。

【互動測驗引導規則】
請根據你「剛剛回答的內容」決定是否發起測驗：

1. **需要詢問的情況**：
   - 當你解釋了一個**知識點、概念、理論**或回答了**學術問題**時。
   - 請在結尾詢問：「這樣說明清楚嗎？要不要我出一題相關的題目來試試看？」

2. **不需要詢問的情況 (禁止囉嗦)**：
   - 當對話僅為**打招呼** (如 "Hi", "你好")。
   - 當對話為**簡單確認** (如 "好", "知道了", "謝謝")。
   - 當使用者**拒絕測驗** (如 "不用了")。

   **在上述情況下，請直接結束回答，不要附加任何測驗詢問。**
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "【歷史對話】：\n{chat_history}\n\n【參考資料】：\n{context}\n\n【學生問題】：\n{question}")
])

# 5. 測驗相關 Prompt
quiz_gen_chain = ChatPromptTemplate.from_messages([
    ("system", "你是一個出題老師。請根據主題：'{topic}'，出一道單選題或簡答題。請列出題目與選項，**不要**給答案。")
]) | llm | StrOutputParser()

quiz_eval_chain = ChatPromptTemplate.from_messages([
    ("system", "你是一個批改老師。題目：'{question}'\n學生回答：'{user_input}'\n請判斷正確性並給予解析。")
]) | llm | StrOutputParser()

# --- 核心函數 ---

def check_user_intent(user_input):
    try:
        result = intent_chain.invoke({"user_input": user_input}).strip()
        for tag in ["WANT_QUIZ", "REFUSE_QUIZ", "OTHER"]:
            if tag in result: return tag
        return "OTHER"
    except: return "OTHER"

def generate_teaching_response(message, history, retriever=None):
    context_text = ""
    source_type = ""

    # RAG
    if retriever:
        try:
            docs = retriever.invoke(message)
            if docs:
                context_text = "\n".join([d.page_content for d in docs])
                source_type = "(📚 講義)"
        except: pass

    # Web Search
    if not context_text:
        web_text = search_web_text(message)
        if web_text:
            context_text = web_text
            source_type = "(🌐 網路)"
        else:
            source_type = "(🤖 內建)"

    # History string
    chat_history_str = ""
    if history:
        for turn in history[-3:]:
            chat_history_str += f"Student: {turn[0]}\nAI: {turn[1]}\n"

    chain = prompt_template | llm | StrOutputParser()
    response = chain.invoke({
        "chat_history": chat_history_str,
        "context": context_text,
        "question": message
    })

    return response + f"\n\n_{source_type}_"

def generate_quiz(topic):
    return quiz_gen_chain.invoke({"topic": topic})

def evaluate_quiz(question, user_input):
    return quiz_eval_chain.invoke({"question": question, "user_input": user_input})

print("✅ Block 3: Prompts (乾淨版) 更新完成！")

## 4. 建置互動介面與狀態機 (Gradio)
這是最後一步，打造具備「狀態管理」的 Web 介面：
1. **對話狀態機 (`chat_handler`)**：實作複雜的互動流程，讓 AI 在「一般問答」、「待命測驗」(詢問意願) 與「測驗進行中」(出題/批改) 三種模式間自動切換。
2. **介面佈局**：設定簡潔的對話視窗與檔案上傳區，專注於文字交流。

In [ ]:
import gradio as gr
import os
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print(f"🔍 目前 Gradio 版本: {gr.__version__}")

def chat_handler(message, history, state):
    if state is None:
        state = {"retriever": None, "mode": "normal", "last_topic": "", "current_quiz": ""}

    mode = state.get("mode", "normal")

    # 1. 先做 LLM 意圖判斷
    intent = check_user_intent(message)

    # 2. 定義強制的同意關鍵字 (保險絲)
    # 這些字出現時，不管 LLM 說什麼，都當作「要測驗」
    direct_agree_keywords = ["好", "要", "ok", "yes", "好啊", "好的", "來", "是", "出題"]
    is_direct_agree = any(x == message.lower().strip() or x in message.lower() for x in direct_agree_keywords)

    # --- 全域觸發測驗 ---
    # 如果使用者說「考考我」，或是意圖明顯是想要測驗
    if mode == "normal" and (intent == "WANT_QUIZ" or "考" in message):
        try:
            topic = state.get("last_topic", "一般常識")
            quiz_content = generate_quiz(topic)
            state["current_quiz"] = quiz_content
            state["mode"] = "quiz_answering"
            return f"👌 沒問題！針對「{topic}」來一題：\n\n{quiz_content}", state
        except:
            return "❌ 出題系統忙碌中...", state

    # --- 狀態 1: 待命測驗 (Quiz_Ask) ---
    if mode == "quiz_ask":
        # 【關鍵修復】: 只要 意圖是WANT_QUIZ  或是  中了關鍵字(好/ok)，就出題
        if intent == "WANT_QUIZ" or is_direct_agree:
            try:
                topic = state.get("last_topic", "剛才的內容")
                quiz_content = generate_quiz(topic)
                state["current_quiz"] = quiz_content
                state["mode"] = "quiz_answering"
                return f"🎉 挑戰開始：\n\n{quiz_content}", state
            except:
                state["mode"] = "normal"
                return f"❌ 出題失敗。", state

        elif intent == "REFUSE_QUIZ":
            state["mode"] = "normal"
            return "好的，那我們繼續討論其他話題。", state

        else:
            state["mode"] = "normal"
            # 視為新問題，繼續往下執行 generate_teaching_response

    # --- 狀態 2: 測驗作答 (Quiz_Answering) ---
    if mode == "quiz_answering":
        if intent == "REFUSE_QUIZ":
             state["mode"] = "normal"
             return "沒關係，我們回到課程吧。", state

        try:
            question = state.get("current_quiz", "")
            feedback = evaluate_quiz(question, message)
            state["mode"] = "normal"
            return f"{feedback}\n\n(測驗結束)", state
        except:
            state["mode"] = "normal"
            return f"❌ 批改錯誤。", state

    # --- 狀態 3: 正常模式 (RAG + Search) ---
    try:
        response = generate_teaching_response(message, history, state.get("retriever"))
        state["last_topic"] = message
        state["mode"] = "quiz_ask"
        return response, state

    except Exception as e:
        return f"❌ 系統錯誤: {e}", state

def on_upload(file, state):
    if not file: return "等待上傳...", state
    if state is None: state = {"retriever": None, "mode": "normal"}

    try:
        file_path = file.name if hasattr(file, 'name') else file
        # 這裡呼叫 Block 2 的函數
        from langchain_community.document_loaders import PyPDFLoader

        retriever = process_uploaded_pdf(file_path)
        if retriever:
            state["retriever"] = retriever
            return "✅ 講義已讀取！優先依據講義回答。", state
        return "❌ 讀取失敗", state
    except Exception as e:
        return f"❌ 上傳錯誤: {e}", state

with gr.Blocks(theme=gr.themes.Soft(), title="全能 AI 家教") as demo:
    state = gr.State({"retriever": None, "mode": "normal", "last_topic": "", "current_quiz": ""})

    gr.Markdown("""
    # 🎓 全能 AI 家教
    ### RAG 文件解讀 | 聯網搜尋 | 混合式意圖判斷
    """)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📂 上傳講義 (PDF)")
            status = gr.Textbox(label="狀態", value="準備就緒", interactive=False)

        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                height=650,
                label="對話視窗",
                avatar_images=(None, "https://cdn-icons-png.flaticon.com/512/4712/4712027.png")
            )
            msg = gr.Textbox(label="輸入", placeholder="例如：解釋供需法則...")
            with gr.Row():
                btn = gr.Button("送出", variant="primary")
                clear = gr.ClearButton([msg, chatbot], value="清除")

    pdf_input.upload(on_upload, [pdf_input, state], [status, state])

    def user(user_message, history):
        return "", history + [[user_message, None]]

    def bot(history, app_state):
        if not history: return history, app_state
        user_message = history[-1][0]
        bot_response, new_state = chat_handler(user_message, history[:-1], app_state)
        history[-1][1] = bot_response
        return history, new_state

    msg.submit(user, [msg, chatbot], [msg, chatbot]).then(bot, [chatbot, state], [chatbot, state])
    btn.click(user, [msg, chatbot], [msg, chatbot]).then(bot, [chatbot, state], [chatbot, state])

print("正在啟動全能家教...")
demo.launch(share=True, debug=True)